# 🏗️ Starter — Evaluasi Model Produksi dari Nol (Bab 8)> Notebook eksperimen. **Prasyarat:** semua 52 test hijau> (`python -m unittest discover -s tests -v`).**Aturan main:** keputusan (threshold, Platt) diambil di CALIB. Test disentuhSEKALI per keputusan untuk melapor — bukan untuk menyetel.

## Setup & Peta Data

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import matplotlib.pyplot as plt
import numpy as np

from prod_mini.calibration import (apply_platt, binned_calibration, ece, fit_platt,
                                   sigmoid)
from prod_mini.data import GAN_MEAN, make_dataset, make_gan_data, split_tiga_arah
from prod_mini.metrics import mae, r2, rmse, skill_score
from prod_mini.models import fit_ham, fit_neg, fit_ols, predict, predict_ham, predict_neg
from prod_mini.persist import load_model, roundtrip_ok, save_model, sha256_file
from prod_mini.threshold import confusion, pilih_threshold, prf

X, y = make_dataset()
Xtr, ytr, Xca, yca, Xte, yte = split_tiga_arah(X, y)
print(f'train={len(Xtr)}  calib={len(Xca)}  test={len(Xte)}')

# label keputusan: 1 jika y > median(train) — didefinisikan dari TRAIN saja
med = float(np.median(ytr))
ytr01 = (ytr > med).astype(int)
yca01 = (yca > med).astype(int)
yte01 = (yte > med).astype(int)
print(f'median(train)={med:.4f} | prevalence: calib {yca01.mean():.3f}, test {yte01.mean():.3f}')


In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(ytr, bins=40, alpha=0.6, label='train')
plt.hist(yte, bins=40, alpha=0.6, label='test')
plt.axvline(med, color='r', ls='--', label=f'median train = {med:.2f}')
plt.xlabel('y'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Distribusi target: resep diketahui, evaluasi harus jujur')
plt.show()


## Bagian 1 — Tiga Kandidat, Satu Baseline, Keputusan ADOPT/TOLAKRMSE tanpa pembanding tidak bermakna. Skill score yang memutuskan.

In [ ]:
# Tiga kandidat model — fit di TRAIN, evaluasi RMSE di TEST
w_ols, b_ols = fit_ols(Xtr, ytr)
w_neg, b_neg = fit_neg(Xtr, ytr)
mu = fit_ham(ytr)

rmse_ols = rmse(yte, predict(w_ols, b_ols, Xte))
rmse_neg = rmse(yte, predict_neg(w_neg, b_neg, Xte))
rmse_ham = rmse(yte, predict_ham(mu, Xte))
r2_ols = r2(yte, predict(w_ols, b_ols, Xte))

print(f'w OLS = {np.round(w_ols, 3).tolist()} (true [2.0, -1.5]), b = {b_ols:.3f} (true 1.0)')
print(f'RMSE  : OLS {rmse_ols:.4f} | neg {rmse_neg:.4f} | HAM {rmse_ham:.4f}')
print(f'R2 OLS: {r2_ols:.4f}')
assert 1.0 < rmse_ols < 1.6, 'RMSE OLS ~1.25 (noise sigma 1.2)'
assert 4.0 < rmse_ham < 5.8
assert rmse_neg > rmse_ham, 'model salah arah harus kalah dari HAM'

# Keputusan ADOPT/TOLAK dari skill score — bukan dari selera
skill_ols = skill_score(rmse_ols, rmse_ham)
skill_neg = skill_score(rmse_neg, rmse_ham)
print(f'skill vs HAM: OLS {skill_ols:+.3f} -> {"ADOPT" if skill_ols > 0 else "TOLAK"}')
print(f'             neg {skill_neg:+.3f} -> {"ADOPT" if skill_neg > 0 else "TOLAK"}')
assert 0.6 < skill_ols < 0.9 and skill_neg < 0
print('✅ Baseline tercatat, keputusan berbasis skill score.')


## Bagian 2 — Persistensi: Hash + Round-trip = Bukti, Bukan Harapan

In [ ]:
# Persistensi + bukti reproducibility
import tempfile

tmpdir = tempfile.mkdtemp()
p1 = f'{tmpdir}/model_v1.npz'
p2 = f'{tmpdir}/model_v1_re_save.npz'
save_model(p1, w_ols, b_ols)
save_model(p2, w_ols, b_ols)
h1, h2 = sha256_file(p1), sha256_file(p2)
print(f'sha256 v1        : {h1[:16]}...')
print(f'sha256 re-save   : {h2[:16]}...')
assert h1 == h2, 'save ulang model yang sama harus menghasilkan byte identik'

pred_sebelum = predict(w_ols, b_ols, Xte)
assert roundtrip_ok(w_ols, b_ols, Xte, pred_sebelum), 'round-trip gagal!'
w2, b2 = load_model(p1)
assert np.array_equal(predict(w2, b2, Xte), pred_sebelum)
print('✅ Round-trip OK: prediksi sebelum & sesudah save/load identik bit-per-bit.')
print('   Artefak ini boleh naik ke serving — dengan BUKTI, bukan harapan.')


## Bagian 3 — Threshold: Kenop Bisnis yang Diputuskan di CALIBTermasuk demo angka bocor yang lebih tinggi — dan kenapa ia TIDAK SAH.

In [ ]:
# Threshold: putuskan di CALIB, evaluasi SEKALI di TEST
kandidat = np.arange(0.0, 3.01, 0.1)
score_ols_ca = predict(w_ols, b_ols, Xca)
score_ols_te = predict(w_ols, b_ols, Xte)
score_ham_te = predict_ham(mu, Xte)

thr_star, cm_c, m_c = pilih_threshold(yca01, score_ols_ca, kandidat)
print(f'thr* = {thr_star:.1f} (dari CALIB)')
print(f'calib: cm={cm_c}')
print(f'       P={m_c["precision"]:.3f} R={m_c["recall"]:.3f} F1={m_c["f1"]:.3f}')
assert 0.8 <= thr_star <= 1.6
assert 0.85 < m_c['f1'] < 0.95

cm_t = confusion(yte01, score_ols_te, thr_star)
m_t = prf(cm_t)
cm_ham = confusion(yte01, score_ham_te, thr_star)
m_ham = prf(cm_ham)
print(f'test @thr*    : cm={cm_t} F1={m_t["f1"]:.3f}')
print(f'test HAM      : cm={cm_ham} F1={m_ham["f1"]:.3f}  <- baseline gagal total')
assert 0.88 < m_t['f1'] < 0.98 and m_ham['f1'] == 0.0

# Demo kebocoran: kalau threshold disetel LANGSUNG di test
thr_leak, _, m_leak = pilih_threshold(yte01, score_ols_te, kandidat)
print(f'\nBocor (tune di test): F1={m_leak["f1"]:.3f} @thr={thr_leak:.1f}')
print(f'Jujur (thr dari calib): F1={m_t["f1"]:.3f} @thr={thr_star:.1f}')
assert m_leak['f1'] >= m_t['f1'], 'angka bocor harus terlihat lebih bagus (itu jebakannya)'
print('✅ Angka bocor lebih tinggi — dan justru karena itu TIDAK SAH.')


## Bagian 4 — Kalibrasi Probability: ECE & Platt Scaling

In [ ]:
# Kalibrasi probability: raw vs Platt (belajar dari CALIB)
p_raw_te = sigmoid(score_ols_te)
a_p, b_p = fit_platt(score_ols_ca, yca01)
p_cal_te = apply_platt(a_p, b_p, score_ols_te)
ece_raw = ece(yte01, p_raw_te, 10)
ece_cal = ece(yte01, p_cal_te, 10)
print(f'ECE raw   = {ece_raw:.4f}')
print(f'ECE platt = {ece_cal:.4f}  (a={a_p:.3f}, b={b_p:.3f})')
assert ece_cal <= ece_raw + 0.01, 'Platt harus memperbaiki (atau setara)'
print('✅ Probabilitas model kini bisa dipercaya untuk keputusan berbasis risiko.')

bins_raw = binned_calibration(yte01, p_raw_te, 10)
bins_cal = binned_calibration(yte01, p_cal_te, 10)
xs = np.arange(10) / 10 + 0.05
plt.figure(figsize=(6.5, 5))
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='sempurna')
plt.bar(xs - 0.18, [b['acc'] for b in bins_raw], width=0.34, alpha=0.6, label='raw')
plt.bar(xs + 0.18, [b['acc'] for b in bins_cal], width=0.34, alpha=0.6, label='platt')
plt.xlabel('confidence bin'); plt.ylabel('accuracy')
plt.legend(); plt.grid(alpha=0.3)
plt.title(f'Reliability diagram — ECE raw {ece_raw:.3f} vs platt {ece_cal:.3f}')
plt.show()


## Bagian 5 — GAN 1D: Dua Pemain Menuju EquilibriumPipeline identik dengan lab Bagian 8; di sini kamu mengamati sebagai 'operator'.

In [ ]:
# data 'nyata' GAN: N(2.0, 0.5^2), seed terkunci
real_data = make_gan_data()
print(f'real: n={len(real_data)} mean={real_data.mean():.4f} std={real_data.std():.4f}')

# pipeline GAN 1D — sama dengan lab Bagian 8
rg = np.random.default_rng(8)
G_MU, G_LOGS = -2.0, np.log(0.5)          # G: x = mu + exp(logs)*z
D_W1, D_B1 = rg.normal(0.0, 0.5, 8), np.zeros(8)   # init nol = saddle point!
D_W2, D_B2 = rg.normal(0.0, 0.5, 8), 0.0
LR = 0.05

def gen_sample(bs, rng_g):
    z = rng_g.normal(0.0, 1.0, bs)
    return z * np.exp(G_LOGS) + G_MU, z

def disc_forward(x):
    x = np.asarray(x, dtype=float)
    h = np.tanh(x[:, None] * D_W1 + D_B1)
    return h @ D_W2 + D_B2, h

def dscore(x):
    return sigmoid(disc_forward(x)[0])

def d_loss(x_real, x_fake):
    return float(-(np.log(dscore(x_real) + 1e-12).mean()
                   + np.log(1.0 - dscore(x_fake) + 1e-12).mean()))

def g_loss(x_fake):
    return float(-np.log(dscore(x_fake) + 1e-12).mean())
print('pipeline GAN siap.')


In [ ]:
# Training GAN 1D — dua pemain, dua gradien, satu equilibrium
rng_t = np.random.default_rng(8)
BATCH, EPOCHS = 128, 300
hist = []
for ep in range(EPOCHS):
    xr = rng_t.choice(real_data, BATCH, replace=False)
    xf, _ = gen_sample(BATCH, rng_t)

    for x, label in ((xr, 1.0), (xf, 0.0)):      # update D
        o, h = disc_forward(x)
        p = sigmoid(o)
        dO = (p - label) / len(x)
        dW2 = h.T @ dO; dB2 = dO.sum()
        dH = dO[:, None] * D_W2[None, :] * (1.0 - h ** 2)
        dW1 = (x[:, None] * dH).sum(0); dB1 = dH.sum(0)
        D_W1 -= LR * dW1; D_B1 -= LR * dB1
        D_W2 -= LR * dW2; D_B2 -= LR * dB2

    xf, _ = gen_sample(BATCH, rng_t)             # update G (melewati D)
    o_g, h_g = disc_forward(xf)
    p_g = sigmoid(o_g)
    dO_g = (p_g - 1.0) / len(xf)
    dx_g = (D_W2[None, :] * (1.0 - h_g ** 2)) @ D_W1
    dmu = float(np.sum(dO_g * dx_g))
    dlogs = float(np.sum(dO_g * dx_g * (xf - G_MU)))
    G_MU -= LR * dmu
    G_LOGS -= LR * dlogs

    if ep % 50 == 0 or ep == EPOCHS - 1:
        xs_, _ = gen_sample(2000, rng_t)
        hist.append((ep, d_loss(xr, xs_), g_loss(xs_),
                     float(xs_.mean()), float(xs_.std())))
        print(f'ep {ep:4d}: D={hist[-1][1]:.3f} G={hist[-1][2]:.3f} '
              f'fake mean={hist[-1][3]:+.3f} std={hist[-1][4]:.3f}')


In [ ]:
# Tanda konvergensi ke equilibrium
fake_final, _ = gen_sample(5000, rng_t)
d_real_final = float(dscore(real_data).mean())
d_fake_final = float(dscore(fake_final).mean())
print(f'G akhir: mu={G_MU:.4f} (target {GAN_MEAN}), sigma={np.exp(G_LOGS):.4f} (target 0.5)')
print(f'D(real)={d_real_final:.3f}  D(fake)={d_fake_final:.3f}  (equilibrium ~0.5 keduanya)')
assert 1.5 < G_MU < 2.8, 'G harus menemukan mu ~2.0'
assert d_fake_final > 0.2 and abs(d_real_final - d_fake_final) < 0.25

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.hist(real_data, bins=40, alpha=0.6, label='real')
plt.hist(fake_final, bins=40, alpha=0.6, label='fake (G)')
plt.legend(); plt.title('Distribusi real vs generated')
plt.subplot(1, 2, 2)
plt.plot([h_[0] for h_ in hist], [h_[3] for h_ in hist], label='fake mean')
plt.axhline(GAN_MEAN, color='r', ls='--', label='target 2.0')
plt.xlabel('epoch'); plt.legend(); plt.title('G belajar mean')
plt.tight_layout(); plt.show()
print('✅ Tidak ada loss yang monoton turun — dua pemain saling menjaga.')


## Bagian 6 — Ringkasan Angka AkhirIsi RUBRIK.md dari angka-angka ini.

In [ ]:
print('=' * 56)
print('RINGKASAN ANGKA AKHIR (isi RUBRIK.md dari sini)')
print('=' * 56)
print(f'1. RMSE test           : OLS {rmse_ols:.3f} | neg {rmse_neg:.3f} | HAM {rmse_ham:.3f}')
print(f'2. skill vs HAM        : OLS {skill_ols:+.3f} (ADOPT) | neg {skill_neg:+.3f} (TOLAK)')
print(f'3. hash deterministik  : {h1[:16]}... (re-save identik)')
print(f'4. F1 test @thr={thr_star:.1f}   : OLS {m_t["f1"]:.3f} | HAM {m_ham["f1"]:.3f}')
print(f'5. ECE                 : raw {ece_raw:.3f} -> platt {ece_cal:.3f}')
print(f'6. GAN equilibrium     : mu={G_MU:.3f} (target 2.0), D(fake)={d_fake_final:.3f}')
print('\nTest disentuh sekali untuk tiap keputusan — angka ini final.')


**Pertanyaan Analisis** (jawab di cell markdown — bagian penilaian, lihat RUBRIK.md):
1. RMSE OLS ~1.25 terdengar bagus — tunjukkan dengan skill score kenapa klaim itu   baru bermakna SETELAH dibandingkan dengan HAM. Model 'negatif' punya RMSE ~6.7:   mengapa keputusan yang benar adalah TOLAK, bukan 'perbaiki dikit'?2. Threshold bocor (tune di test) menghasilkan F1 lebih tinggi daripada jujur.   Jelaskan MEKANISME kebocorannya, dan kenapa selisihnya kecil di dataset ini   tidak membuatnya sah untuk produksi.3. Round-trip test lolos, tapi tim masih meng-claim 'model sudah di-update' tanpa   hash. Apa skenario kegagalan yang terlewat, dan bagaimana hash menutupnya?4. HAM gagal total sebagai klasifikator (F1=0) tapi berguna sebagai baseline regresi.   Jelaskan dua peran berbeda itu, dan kapan baseline 'tidak berdaya' justru   menunjukkan metrikmu yang salah?5. ECE turun setelah Platt scaling. Di keputusan bisnis apa penurunan ECE ini   benar-benar penting, dan kapan F1 tinggi TANPA kalibrasi justru cukup?6. GAN berakhir di equilibrium, bukan 'menang'. Kaitkan dengan mode collapse:   kalau G punya kapasitas jauh lebih besar dari variasi data, apa yang terjadi   dan bagaimana kamu mendeteksinya dari histogram/diagnostik?

### (Opsional) Eksperimen untuk Pertanyaan 2 — seberapa besar umpan bocor?Bandingkan distribusi F1 'bocor' vs 'jujur' di banyak bootstraps kecil dari CALIB.Perhatikan: bocor hampir selalu menang — bias sistematis, bukan noise.

In [ ]:
# TODO: jalankan klausul kecil ini di CALIB (bukan test)
# thrs = np.arange(0.0, 3.01, 0.1)
# f1_calib = pilih_threshold(yca01, score_ols_ca, thrs)[2]['f1']
# print(f'F1 di calib sendiri = {f1_calib:.3f} — ini yang dijanjikan model.')
# # lalu cek gap ke test:
# print(f'gap test - calib = {m_t["f1"] - f1_calib:+.3f}')
